# 面试问题：怎样手写 CNN 与 ResNet，并说明残差连接为什么有效？

## 可直接复述的回答主线

1. CNN 用共享卷积核提取局部模式，通过层叠卷积与下采样逐步扩大感受野，再用分类头输出 logits。
2. ResNet 基本块学习残差 F(x)，输出 F(x)+shortcut(x)，让网络至少能保留恒等信息通路。
3. 当通道数或空间尺寸变化时，shortcut 必须用 1×1 卷积和相同 stride 投影，才能逐元素相加。
4. 残差连接改善的是优化路径而不是自动保证泛化；初始化、归一化、数据增强和训练日程仍然关键。
5. 评测应在相同图像上比较简单统计基线、普通 CNN 与 ResNet，并展示特征 shape、残差幅度、梯度和逐样本预测。
6. 故障排查要区分主分支学不到、shortcut shape 错误、BatchNorm 模式错误和输入分布漂移。
7. 生产还需真实缺陷图、类别长尾、增强、校准、目标硬件延迟、量化和鲁棒性验证。

下面用同一批可读输入依次验证朴素基线、手写核心机制、中间过程、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是 12 张 16×16 脱敏产线表面检测小图，三类分别为横向划痕、纵向划痕和块状污染，每类 4 张且位置有轻微偏移。每张前景像素数和全局均值被控制为相同，因此只看亮度的基线无法识别空间形状；这些合成图只用于解释卷积与残差计算。

In [1]:
import math  # 汇总训练梯度并计算准确率。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦图像实验。
import torch  # 使用基础卷积层和自动微分手写 CNN 与 ResNet。
torch.manual_seed(221)  # 固定模型初始化与训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程提高可复现性。
class_names = ["横向划痕", "纵向划痕", "块状污染"]  # 定义三种产线缺陷类别。
yy, xx = torch.meshgrid(torch.arange(16), torch.arange(16), indexing="ij")  # 构造像素坐标网格。
images = []  # 保存十二张单通道输入图。
labels = []  # 保存每张图的缺陷标签。
sample_ids = []  # 保存可读图像编号。
for class_index in range(3):  # 依次生成三个空间模式类别。
    for variant in range(4):  # 为每类生成四个位置变体。
        foreground = torch.zeros(16, 16)  # 初始化当前二值缺陷区域。
        offset = variant - 2  # 把变体编号映射为轻微位置偏移。
        if class_index == 0:  # 构造两像素高的横向划痕。
            row = 7 + offset  # 计算当前横线中心位置。
            foreground[row:row + 2, :] = 1.0  # 写入三十二个横向前景像素。
        elif class_index == 1:  # 构造两像素宽的纵向划痕。
            column = 7 + offset  # 计算当前竖线中心位置。
            foreground[:, column:column + 2] = 1.0  # 写入三十二个纵向前景像素。
        else:  # 构造四乘八的块状污染。
            top = 5 + offset  # 计算当前污染块顶部位置。
            foreground[top:top + 4, 4:12] = 1.0  # 写入同样三十二个块状前景像素。
        texture = 0.03 * torch.sin((yy + variant) * 0.7) * torch.cos((xx + class_index) * 0.5)  # 加入可复现的弱表面纹理。
        image = 0.08 + 0.75 * foreground + texture  # 合成背景、缺陷响应和纹理。
        image = image - image.mean() + 0.20  # 强制所有图全局均值相同以限制亮度基线。
        images.append(image.unsqueeze(0))  # 保存单通道图像。
        labels.append(class_index)  # 保存当前类别编号。
        sample_ids.append(f"defect-{class_index}-{variant}")  # 生成稳定样本 ID。
images = torch.stack(images)  # 堆叠为十二乘一乘十六乘十六张量。
labels = torch.tensor(labels, dtype=torch.long)  # 转换为交叉熵标签张量。
def render_image(image):  # 把灰度小图渲染为可读字符画。
    return "\n".join("".join("#" if float(pixel) > 0.5 else "." for pixel in row) for row in image.squeeze(0))  # 用阈值生成十六行字符。
print("教学实验输入：产线缺陷图，shape=", tuple(images.shape), "means=", torch.round(images.mean(dim=(1, 2, 3)) * 10000) / 10000)  # 展示图像规模和相同均值合同。
for index in range(len(images)):  # 逐样本展示 ID、标签和字符图。
    print(f"\n{sample_ids[index]} label={class_names[labels[index]]}\n{render_image(images[index])}")  # 输出当前缺陷的真实空间模式。

教学实验输入：产线缺陷图，shape= (12, 1, 16, 16) means= tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
        0.2000, 0.2000, 0.2000])

defect-0-0 label=横向划痕
................
................
................
................
................
################
################
................
................
................
................
................
................
................
................
................

defect-0-1 label=横向划痕
................
................
................
................
................
................
################
################
................
................
................
................
................
................
................
................

defect-0-2 label=横向划痕
................
................
................
................
................
................
................
################
################
................
................
................
................
..........

## 2. Baseline / 基线：只用全局平均亮度做最近类均值

三类图像的前景像素数相同，并被校正到同一全局均值。亮度基线完全丢掉空间排列，分数并列时固定选第一类，因此只能达到类别先验水平。

In [2]:
global_means = images.mean(dim=(1, 2, 3))  # 为每张图提取唯一的全局亮度特征。
class_mean_brightness = torch.stack([global_means[labels == class_index].mean() for class_index in range(3)])  # 计算三类训练均值亮度。
baseline_distances = torch.abs(global_means[:, None] - class_mean_brightness[None, :])  # 计算每张图到三类亮度中心的距离。
baseline_predictions = baseline_distances.argmin(dim=1)  # 对并列距离按索引选择第一类。
baseline_accuracy = float((baseline_predictions == labels).to(torch.float32).mean().item())  # 计算同一十二张图的亮度基线准确率。
print("Baseline：全局平均亮度最近中心")  # 标记下表为空间信息被抹除的结果。
print("sample       true       mean      distances                 prediction")  # 输出逐图基线表头。
for index in range(len(images)):  # 逐图展示全局统计分类过程。
    print(f"{sample_ids[index]:<12} {class_names[labels[index]]:<8} {global_means[index].item():.5f} {baseline_distances[index].tolist()} {class_names[baseline_predictions[index]]}")  # 输出真实类、亮度、距离和预测。
print(f"Baseline accuracy={baseline_accuracy:.4f}")  # 展示丢弃空间结构后的类别先验水平。

Baseline：全局平均亮度最近中心
sample       true       mean      distances                 prediction
defect-0-0   横向划痕     0.20000 [1.4901161193847656e-08, 1.4901161193847656e-08, 1.4901161193847656e-08] 横向划痕
defect-0-1   横向划痕     0.20000 [1.4901161193847656e-08, 1.4901161193847656e-08, 1.4901161193847656e-08] 横向划痕
defect-0-2   横向划痕     0.20000 [0.0, 2.9802322387695312e-08, 2.9802322387695312e-08] 横向划痕
defect-0-3   横向划痕     0.20000 [0.0, 2.9802322387695312e-08, 2.9802322387695312e-08] 横向划痕
defect-1-0   纵向划痕     0.20000 [2.9802322387695312e-08, 0.0, 0.0] 纵向划痕
defect-1-1   纵向划痕     0.20000 [1.4901161193847656e-08, 1.4901161193847656e-08, 1.4901161193847656e-08] 横向划痕
defect-1-2   纵向划痕     0.20000 [1.4901161193847656e-08, 1.4901161193847656e-08, 1.4901161193847656e-08] 横向划痕
defect-1-3   纵向划痕     0.20000 [0.0, 2.9802322387695312e-08, 2.9802322387695312e-08] 横向划痕
defect-2-0   块状污染     0.20000 [2.9802322387695312e-08, 0.0, 0.0] 纵向划痕
defect-2-1   块状污染     0.20000 [1.4901161193847656e-08, 1.4901161193847

## 3. 底层实现：普通 CNN、ResidualBlock 与投影 Shortcut

不导入任何现成架构。普通 CNN 直接串联卷积；ResidualBlock 显式计算主分支和 shortcut，当 stride 或通道变化时使用 1×1 投影。两个模型在完全相同的十二张图上真实训练。

In [3]:
class PlainCNN(torch.nn.Module):  # 定义无残差的普通卷积分类器。
    def __init__(self, class_count=3):  # 初始化三段卷积和分类头。
        super().__init__()  # 注册 PyTorch 子模块。
        self.features = torch.nn.Sequential(torch.nn.Conv2d(1, 8, 3, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(2), torch.nn.Conv2d(8, 16, 3, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(2), torch.nn.Conv2d(16, 24, 3, padding=1), torch.nn.ReLU())  # 串联局部卷积和两次下采样。
        self.head = torch.nn.Linear(24, class_count)  # 把全局池化特征映射为类别 logits。
    def forward(self, inputs, return_debug=False):  # 执行普通 CNN 前向并可返回中间特征。
        features = self.features(inputs)  # 提取四乘四空间特征图。
        pooled = features.mean(dim=(2, 3))  # 对空间维执行全局平均池化。
        logits = self.head(pooled)  # 生成三类未归一化分数。
        return (logits, {"features": features, "pooled": pooled}) if return_debug else logits  # 按需返回解释张量。
class ResidualBlock(torch.nn.Module):  # 定义两层卷积的基础残差块。
    def __init__(self, input_channels, output_channels, stride=1):  # 初始化主分支和条件投影 shortcut。
        super().__init__()  # 注册卷积与归一化参数。
        self.conv_one = torch.nn.Conv2d(input_channels, output_channels, 3, stride=stride, padding=1, bias=False)  # 主分支第一层负责可选下采样。
        self.norm_one = torch.nn.BatchNorm2d(output_channels)  # 稳定第一层通道统计。
        self.conv_two = torch.nn.Conv2d(output_channels, output_channels, 3, padding=1, bias=False)  # 主分支第二层保持空间尺寸。
        self.norm_two = torch.nn.BatchNorm2d(output_channels)  # 稳定残差输出尺度。
        needs_projection = stride != 1 or input_channels != output_channels  # 判断恒等张量能否直接相加。
        self.shortcut = torch.nn.Sequential(torch.nn.Conv2d(input_channels, output_channels, 1, stride=stride, bias=False), torch.nn.BatchNorm2d(output_channels)) if needs_projection else torch.nn.Identity()  # 为 shape 变化创建 1×1 投影。
    def residual_branch(self, inputs):  # 单独计算 F(x) 供失败实验观察。
        hidden = torch.relu(self.norm_one(self.conv_one(inputs)))  # 执行第一层卷积、归一化和激活。
        return self.norm_two(self.conv_two(hidden))  # 返回激活前的残差分支输出。
    def forward(self, inputs, return_debug=False):  # 显式计算 F(x)+S(x)。
        residual = self.residual_branch(inputs)  # 取得主分支学习到的残差。
        identity = self.shortcut(inputs)  # 取得恒等或投影 shortcut。
        output = torch.relu(residual + identity)  # 相加后执行块输出激活。
        return (output, {"residual": residual, "identity": identity}) if return_debug else output  # 按需返回两条路径证据。
class MiniResNet(torch.nn.Module):  # 定义适合十六像素输入的手写小型 ResNet。
    def __init__(self, class_count=3):  # 初始化 stem、三个残差块和分类头。
        super().__init__()  # 注册完整网络参数。
        self.stem = torch.nn.Sequential(torch.nn.Conv2d(1, 8, 3, padding=1, bias=False), torch.nn.BatchNorm2d(8), torch.nn.ReLU())  # 将单通道图映射到八通道。
        self.block_one = ResidualBlock(8, 8)  # 在十六乘十六尺度学习恒等残差。
        self.block_two = ResidualBlock(8, 16, stride=2)  # 下采样到八乘八并投影 shortcut。
        self.block_three = ResidualBlock(16, 16)  # 在八乘八尺度继续学习残差。
        self.head = torch.nn.Linear(16, class_count)  # 把全局特征映射到三类 logits。
    def forward(self, inputs, return_debug=False):  # 执行 stem、残差堆叠和分类。
        stem = self.stem(inputs)  # 生成第一层局部边缘特征。
        first, first_debug = self.block_one(stem, return_debug=True)  # 执行同尺寸残差块并保留路径。
        second, second_debug = self.block_two(first, return_debug=True)  # 执行带投影的下采样残差块。
        third = self.block_three(second)  # 执行最后同尺寸残差块。
        pooled = third.mean(dim=(2, 3))  # 对八乘八特征执行全局平均池化。
        logits = self.head(pooled)  # 生成三类输出分数。
        debug = {"stem": stem, "first": first, "second": second, "third": third, "first_paths": first_debug, "second_paths": second_debug}  # 汇总各级 shape 与残差路径。
        return (logits, debug) if return_debug else logits  # 按需返回中间张量。
def train_classifier(model, steps, learning_rate):  # 在同一十二张图上执行真实分类训练。
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # 创建当前模型独立优化器。
    history = []  # 保存损失、准确率和梯度轨迹。
    for step in range(steps):  # 重复小批次前向与更新。
        model.train()  # 启用 BatchNorm 训练统计。
        optimizer.zero_grad(set_to_none=True)  # 清除上一步参数梯度。
        logits = model(images)  # 对十二张图执行当前架构前向。
        loss = torch.nn.functional.cross_entropy(logits, labels)  # 计算三类交叉熵。
        loss.backward()  # 对卷积、归一化和分类头执行反向传播。
        gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总非空梯度二范数。
        optimizer.step()  # 应用 Adam 更新当前模型参数。
        if step % (steps // 3) == 0 or step == steps - 1:  # 记录四个关键训练时刻。
            accuracy = float((logits.argmax(dim=1) == labels).to(torch.float32).mean().item())  # 计算当前训练批准确率。
            history.append({"step": step, "loss": loss.item(), "accuracy": accuracy, "gradient_norm": gradient_norm})  # 保存可读训练状态。
    model.eval()  # 切换为确定性评估模式。
    return history  # 返回当前架构完整训练证据。
plain_model = PlainCNN()  # 创建无 shortcut 的普通 CNN。
resnet_model = MiniResNet()  # 创建手写残差分类器。
plain_history = train_classifier(plain_model, 180, 0.02)  # 真实训练普通 CNN 供架构对照。
resnet_history = train_classifier(resnet_model, 180, 0.02)  # 在相同数据和步数训练 ResNet。
with torch.no_grad():  # 取得最终 logits 与 ResNet 中间特征。
    plain_logits, plain_debug = plain_model(images, return_debug=True)  # 前向普通 CNN 并读取四乘四特征。
    resnet_logits, resnet_debug = resnet_model(images, return_debug=True)  # 前向 ResNet 并读取三层残差特征。
print("PlainCNN训练轨迹=", plain_history)  # 展示普通网络真实 loss 与梯度。
print("MiniResNet训练轨迹=", resnet_history)  # 展示残差网络真实 loss 与梯度。
print("ResNet shape trace=", {name: tuple(value.shape) for name, value in resnet_debug.items() if isinstance(value, torch.Tensor)})  # 展示 stem 和各块实际空间尺寸。
print("block2 residual/shortcut平均绝对值=", resnet_debug["second_paths"]["residual"].abs().mean().item(), resnet_debug["second_paths"]["identity"].abs().mean().item())  # 展示投影块两条路径的数值规模。

PlainCNN训练轨迹= [{'step': 0, 'loss': 1.1037081480026245, 'accuracy': 0.3333333432674408, 'gradient_norm': 0.07225268570073046}, {'step': 60, 'loss': 1.1126167009933852e-06, 'accuracy': 1.0, 'gradient_norm': 0.00013800792812090952}, {'step': 120, 'loss': 4.5696847905674076e-07, 'accuracy': 1.0, 'gradient_norm': 5.190388902568113e-05}, {'step': 179, 'loss': 3.27825318890973e-07, 'accuracy': 1.0, 'gradient_norm': 3.501075636524504e-05}]
MiniResNet训练轨迹= [{'step': 0, 'loss': 1.277858853340149, 'accuracy': 0.3333333432674408, 'gradient_norm': 2.6396567433159}, {'step': 60, 'loss': 0.00012671078729908913, 'accuracy': 1.0, 'gradient_norm': 0.00065129862393606}, {'step': 120, 'loss': 8.145604078890756e-05, 'accuracy': 1.0, 'gradient_norm': 0.0004218834483486487}, {'step': 179, 'loss': 6.055630001355894e-05, 'accuracy': 1.0, 'gradient_norm': 0.0003187464618357673}]
ResNet shape trace= {'stem': (12, 8, 16, 16), 'first': (12, 8, 16, 16), 'second': (12, 16, 8, 8), 'third': (12, 16, 8, 8)}
block2 resi

## 4. 逐图分类结果与结果解读

三种方案使用完全相同的十二张图。逐样本输出 global-mean 基线、PlainCNN 和 MiniResNet 的预测及 ResNet 置信度，说明模型确实读取了空间形状。

In [4]:
plain_predictions = plain_logits.argmax(dim=1)  # 取得普通 CNN 的逐图预测。
resnet_probabilities = torch.softmax(resnet_logits, dim=1)  # 把 ResNet logits 转为可读类别概率。
resnet_predictions = resnet_probabilities.argmax(dim=1)  # 取得 ResNet 的逐图最高概率类别。
plain_accuracy = float((plain_predictions == labels).to(torch.float32).mean().item())  # 计算普通 CNN 同数据准确率。
resnet_accuracy = float((resnet_predictions == labels).to(torch.float32).mean().item())  # 计算 ResNet 同数据准确率。
print("sample       true       brightness   PlainCNN    MiniResNet  confidence")  # 输出逐图架构对照表头。
for index in range(len(images)):  # 逐图展示三种方法的实际判断。
    confidence = float(resnet_probabilities[index, resnet_predictions[index]].item())  # 读取当前 ResNet 最大类别概率。
    print(f"{sample_ids[index]:<12} {class_names[labels[index]]:<8} {class_names[baseline_predictions[index]]:<12} {class_names[plain_predictions[index]]:<11} {class_names[resnet_predictions[index]]:<11} {confidence:.4f}")  # 输出真实类、三种预测和置信度。
print(f"结果解读：全局亮度baseline={baseline_accuracy:.4f}，PlainCNN={plain_accuracy:.4f}，MiniResNet={resnet_accuracy:.4f}；空间卷积恢复了方向与形状信息。")  # 解释同数据对照结果及受控边界。

sample       true       brightness   PlainCNN    MiniResNet  confidence
defect-0-0   横向划痕     横向划痕         横向划痕        横向划痕        0.9999
defect-0-1   横向划痕     横向划痕         横向划痕        横向划痕        1.0000
defect-0-2   横向划痕     横向划痕         横向划痕        横向划痕        1.0000
defect-0-3   横向划痕     横向划痕         横向划痕        横向划痕        1.0000
defect-1-0   纵向划痕     纵向划痕         纵向划痕        纵向划痕        0.9999
defect-1-1   纵向划痕     横向划痕         纵向划痕        纵向划痕        1.0000
defect-1-2   纵向划痕     横向划痕         纵向划痕        纵向划痕        1.0000
defect-1-3   纵向划痕     横向划痕         纵向划痕        纵向划痕        1.0000
defect-2-0   块状污染     纵向划痕         块状污染        块状污染        0.9999
defect-2-1   块状污染     横向划痕         块状污染        块状污染        0.9999
defect-2-2   块状污染     纵向划痕         块状污染        块状污染        0.9999
defect-2-3   块状污染     横向划痕         块状污染        块状污染        0.9999
结果解读：全局亮度baseline=0.4167，PlainCNN=1.0000，MiniResNet=1.0000；空间卷积恢复了方向与形状信息。


## 5. 失败案例与修正：残差主分支暂时为零时删除 shortcut

将一个残差块的卷积参数全部置零，模拟训练初期或故障分支没有有效输出。若错误地只返回 F(x)，正输入变成全零且输入梯度为零；保留 `F(x)+x` 后，信息和梯度仍沿恒等路径通过。

In [5]:
zero_block = ResidualBlock(4, 4)  # 创建输入输出 shape 相同的残差块。
with torch.no_grad():  # 在无梯度环境构造确定性零主分支。
    for parameter in zero_block.parameters():  # 遍历卷积和 BatchNorm 参数。
        parameter.zero_()  # 把主分支可训练参数全部清零。
zero_block.eval()  # 使用固定 BatchNorm 统计避免批次变化。
broken_input = torch.ones(1, 4, 5, 5, requires_grad=True)  # 创建用于错误路径的正值输入。
broken_output = torch.relu(zero_block.residual_branch(broken_input))  # 错误地删除 shortcut 只保留零 F(x)。
broken_output.sum().backward()  # 对错误输出反向传播到输入。
broken_gradient_norm = float(broken_input.grad.norm().item())  # 记录错误路径输入梯度大小。
corrected_input = torch.ones(1, 4, 5, 5, requires_grad=True)  # 创建完全相同的修正路径输入。
corrected_output = zero_block(corrected_input)  # 使用 F(x)+identity 的正确残差 forward。
corrected_output.sum().backward()  # 沿恒等 shortcut 反向传播。
corrected_gradient_norm = float(corrected_input.grad.norm().item())  # 记录 shortcut 保留的输入梯度。
print(f"错误行为：只返回F(x)，output_mean={broken_output.mean().item():.3f}，input_grad_norm={broken_gradient_norm:.3f}")  # 展示信息和梯度同时消失。
print(f"修正行为：返回F(x)+x，output_mean={corrected_output.mean().item():.3f}，input_grad_norm={corrected_gradient_norm:.3f}")  # 展示恒等路径保留信号与梯度。

错误行为：只返回F(x)，output_mean=0.000，input_grad_norm=0.000
修正行为：返回F(x)+x，output_mean=1.000，input_grad_norm=10.000


## 6. 生产边界

十二张规则图会被模型记忆。真实缺陷系统需要相机与批次切分、光照/模糊/压缩增强、长尾采样、难负例回流、校准与拒识、目标设备上的 batch=1 延迟和显存基准、ONNX/量化数值对齐，以及按产线和材质监控召回率与输入漂移。

In [6]:
cnn_resnet_diagnostics = {"images": len(images), "resolution": tuple(images.shape[-2:]), "baseline_accuracy": baseline_accuracy, "plain_accuracy": plain_accuracy, "resnet_accuracy": resnet_accuracy, "plain_final_loss": plain_history[-1]["loss"], "resnet_final_loss": resnet_history[-1]["loss"], "broken_input_gradient": broken_gradient_norm, "shortcut_input_gradient": corrected_gradient_norm}  # 汇总数据、训练、分类和失败修复指标。
print("生产监控快照：", cnn_resnet_diagnostics)  # 输出视觉分类服务应持续观察的信号。

生产监控快照： {'images': 12, 'resolution': (16, 16), 'baseline_accuracy': 0.4166666567325592, 'plain_accuracy': 1.0, 'resnet_accuracy': 1.0, 'plain_final_loss': 3.27825318890973e-07, 'resnet_final_loss': 6.055630001355894e-05, 'broken_input_gradient': 0.0, 'shortcut_input_gradient': 10.0}


## 7. 最小回归测试

最后一格只保护图像规模、真实训练、分类收益、投影 shape 与 shortcut 梯度。

In [7]:
assert len(images) >= 6 and images.shape == (12, 1, 16, 16)  # 保证包含足够多可显示图像。
assert plain_history[-1]["loss"] < plain_history[0]["loss"] and resnet_history[-1]["loss"] < resnet_history[0]["loss"]  # 保证两种架构均真实优化。
assert all(row["gradient_norm"] > 0.0 for row in plain_history + resnet_history)  # 保证卷积参数获得非零梯度。
assert plain_accuracy > baseline_accuracy and resnet_accuracy > baseline_accuracy and resnet_accuracy >= 0.90  # 保证空间模型优于全局亮度基线。
assert resnet_debug["second_paths"]["residual"].shape == resnet_debug["second_paths"]["identity"].shape  # 保证下采样投影 shortcut 与主分支 shape 一致。
assert broken_gradient_norm == 0.0 and corrected_gradient_norm > 0.0  # 保证删除 shortcut 的梯度失败可复现并修正。